In [29]:
import os, time, zipfile
import duckdb
import pandas as pd
from IPython.display import display

BASE = "D:/Big Data"
ZIP_FILE = f"{BASE}/airline.csv.shuffle.zip"
CSV_FILE = f"{BASE}/airline.csv.shuffle"
OUT = f"{BASE}/outputs_sql"
DB_FILE = f"{OUT}/airline_sql.duckdb"
RAW = "airline_raw"
CLEAN = "airline_clean"

os.makedirs(OUT, exist_ok=True)
os.makedirs(f"{OUT}/tmp", exist_ok=True)

con = duckdb.connect(DB_FILE)
con.execute(f"SET temp_directory = '{OUT}/tmp'")
con.execute("SET preserve_insertion_order = false")

RESULTS = []

def run(name, sql):
    t0 = time.perf_counter()
    out = con.execute(sql).df()
    RESULTS.append({"Stage": name, "Seconds": round(time.perf_counter() - t0, 3)})
    print(f"{name}  {RESULTS[-1]['Seconds']:,.2f} s")
    return out

In [30]:
DISK_BENCH_GB = 2.0
BLOCK = 8 * 1024 * 1024

t0 = time.perf_counter()

limit_bytes = int(min(DISK_BENCH_GB * 1024 ** 3, os.path.getsize(ZIP_FILE)))
read_bytes = 0
with open(ZIP_FILE, "rb", buffering=0) as fh:
    while read_bytes < limit_bytes:
        block = fh.read(min(BLOCK, limit_bytes - read_bytes))
        if not block:
            break
        read_bytes += len(block)

RESULTS.append({"Stage": "0. Disk read benchmark (raw bytes)", "Seconds": round(time.perf_counter() - t0, 3)})
print(f"{read_bytes / 1024 ** 3:.2f} GB read in {RESULTS[-1]['Seconds']:,.2f} s")

2.00 GB read in 0.65 s


In [31]:
t0 = time.perf_counter()

if not os.path.exists(CSV_FILE):
    with zipfile.ZipFile(ZIP_FILE) as z:
        inner = [n for n in z.namelist() if not n.endswith("/")][0]
        with z.open(inner) as src, open(CSV_FILE, "wb") as dst:
            while True:
                block = src.read(64 * 1024 * 1024)
                if not block:
                    break
                dst.write(block)

con.execute(f"""
CREATE OR REPLACE TABLE {RAW} AS
SELECT Year, Month, DayofMonth, DayOfWeek,
       UniqueCarrier, Origin, Dest, Distance,
       DepDelay, ArrDelay,
       Cancelled, Diverted, CancellationCode,
       CarrierDelay, WeatherDelay, NASDelay, SecurityDelay, LateAircraftDelay
FROM read_csv(
    '{CSV_FILE}',
    header = true,
    nullstr = 'NA',
    encoding = 'latin-1',
    types = {{
        'Year': 'INTEGER', 'Month': 'INTEGER', 'DayofMonth': 'INTEGER', 'DayOfWeek': 'INTEGER',
        'UniqueCarrier': 'VARCHAR', 'Origin': 'VARCHAR', 'Dest': 'VARCHAR', 'CancellationCode': 'VARCHAR',
        'Distance': 'DOUBLE', 'DepDelay': 'DOUBLE', 'ArrDelay': 'DOUBLE',
        'Cancelled': 'DOUBLE', 'Diverted': 'DOUBLE',
        'CarrierDelay': 'DOUBLE', 'WeatherDelay': 'DOUBLE', 'NASDelay': 'DOUBLE',
        'SecurityDelay': 'DOUBLE', 'LateAircraftDelay': 'DOUBLE'
    }}
)
""")

RESULTS.append({"Stage": "1. Data loading", "Seconds": round(time.perf_counter() - t0, 3)})

n_raw = con.execute(f"SELECT count(*) c FROM {RAW}").fetchone()[0]
print(f"{n_raw:,} rows loaded in {RESULTS[-1]['Seconds']:,.2f} s")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

123,534,969 rows loaded in 116.97 s


In [32]:
audit_wide = run("2a. Data quality audit (pre-clean)", f"""
SELECT count(*) AS n_rows,
       count(Year) AS Year, count(Month) AS Month,
       count(DayofMonth) AS DayofMonth, count(DayOfWeek) AS DayOfWeek,
       count(UniqueCarrier) AS UniqueCarrier, count(Origin) AS Origin, count(Dest) AS Dest,
       count(Distance) AS Distance, count(DepDelay) AS DepDelay, count(ArrDelay) AS ArrDelay,
       count(Cancelled) AS Cancelled, count(Diverted) AS Diverted,
       count(CancellationCode) AS CancellationCode,
       count(CarrierDelay) AS CarrierDelay, count(WeatherDelay) AS WeatherDelay,
       count(NASDelay) AS NASDelay, count(SecurityDelay) AS SecurityDelay,
       count(LateAircraftDelay) AS LateAircraftDelay
FROM {RAW}
""")

n_rows = int(audit_wide["n_rows"][0])
audit = audit_wide.drop(columns=["n_rows"]).T.reset_index()
audit.columns = ["Column", "Non_null"]
audit["Missing"] = n_rows - audit["Non_null"]
audit["Missing_pct"] = (100 * audit["Missing"] / n_rows).round(3)
display(audit.sort_values("Missing", ascending=False).reset_index(drop=True))

2a. Data quality audit (pre-clean)  0.61 s


,Column,Non_null,Missing,Missing_pct
0,NASDelay,34205536,89329433,72.311
1,WeatherDelay,34205536,89329433,72.311
2,CarrierDelay,34205536,89329433,72.311
3,LateAircraftDelay,34205536,89329433,72.311
4,SecurityDelay,34205536,89329433,72.311
5,CancellationCode,39690529,83844440,67.871
6,ArrDelay,120947440,2587529,2.095
7,DepDelay,121232833,2302136,1.864
8,Distance,123332969,202000,0.164
9,Year,123534969,0,0.000


In [33]:
t0 = time.perf_counter()

con.execute(f"""
CREATE OR REPLACE TABLE {CLEAN} AS
SELECT * EXCLUDE (Cancelled, Diverted),
       (ArrDelay > 15) AS Delayed15
FROM (
    SELECT DISTINCT
           Year, Month, DayofMonth, DayOfWeek,
           UniqueCarrier, Origin, Dest, Distance,
           DepDelay, ArrDelay,
           Cancelled, Diverted, CancellationCode,
           CarrierDelay, WeatherDelay, NASDelay, SecurityDelay, LateAircraftDelay
    FROM {RAW}
    WHERE Year BETWEEN 1987 AND 2030
      AND Month BETWEEN 1 AND 12
      AND Cancelled = 0
      AND Diverted = 0
      AND ArrDelay IS NOT NULL
      AND abs(ArrDelay) <= 1440
      AND abs(coalesce(DepDelay, 0)) <= 1440
) t
""")

RESULTS.append({"Stage": "2b. Cleaning", "Seconds": round(time.perf_counter() - t0, 3)})

n_clean = con.execute(f"SELECT count(*) c FROM {CLEAN}").fetchone()[0]
print(f"{n_clean:,} rows kept in {RESULTS[-1]['Seconds']:,.2f} s")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

118,503,331 rows kept in 327.17 s


In [34]:
schema = run("3a. Schema description", f"""
WITH n AS (SELECT count(*) AS n_rows FROM {CLEAN}),
counts AS (
    SELECT 'Year' AS column_name, count(Year) AS non_null FROM {CLEAN}
    UNION ALL SELECT 'Month', count(Month) FROM {CLEAN}
    UNION ALL SELECT 'DayofMonth', count(DayofMonth) FROM {CLEAN}
    UNION ALL SELECT 'DayOfWeek', count(DayOfWeek) FROM {CLEAN}
    UNION ALL SELECT 'UniqueCarrier', count(UniqueCarrier) FROM {CLEAN}
    UNION ALL SELECT 'Origin', count(Origin) FROM {CLEAN}
    UNION ALL SELECT 'Dest', count(Dest) FROM {CLEAN}
    UNION ALL SELECT 'Distance', count(Distance) FROM {CLEAN}
    UNION ALL SELECT 'DepDelay', count(DepDelay) FROM {CLEAN}
    UNION ALL SELECT 'ArrDelay', count(ArrDelay) FROM {CLEAN}
    UNION ALL SELECT 'CancellationCode', count(CancellationCode) FROM {CLEAN}
    UNION ALL SELECT 'CarrierDelay', count(CarrierDelay) FROM {CLEAN}
    UNION ALL SELECT 'WeatherDelay', count(WeatherDelay) FROM {CLEAN}
    UNION ALL SELECT 'NASDelay', count(NASDelay) FROM {CLEAN}
    UNION ALL SELECT 'SecurityDelay', count(SecurityDelay) FROM {CLEAN}
    UNION ALL SELECT 'LateAircraftDelay', count(LateAircraftDelay) FROM {CLEAN}
    UNION ALL SELECT 'Delayed15', count(Delayed15) FROM {CLEAN}
)
SELECT c.column_name, d.column_type, c.non_null,
       n.n_rows - c.non_null AS missing,
       round(100.0 * (n.n_rows - c.non_null) / n.n_rows, 3) AS missing_pct
FROM counts c
JOIN (DESCRIBE {CLEAN}) d USING (column_name)
CROSS JOIN n
ORDER BY c.column_name
""")
display(schema)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

3a. Schema description  2.75 s


,column_name,column_type,non_null,missing,missing_pct
0,ArrDelay,DOUBLE,118503331,0,0.000
1,CancellationCode,VARCHAR,38354965,80148366,67.634
2,CarrierDelay,DOUBLE,33063650,85439681,72.099
3,DayOfWeek,INTEGER,118503331,0,0.000
4,DayofMonth,INTEGER,118503331,0,0.000
5,Delayed15,BOOLEAN,118503331,0,0.000
6,DepDelay,DOUBLE,118503331,0,0.000
7,Dest,VARCHAR,118503331,0,0.000
8,Distance,DOUBLE,118309103,194228,0.164
9,LateAircraftDelay,DOUBLE,33063650,85439681,72.099


In [35]:
describe = run("3b. Summary statistics (describe)", f"""
SELECT 'Year' AS column_name, count(Year) AS n, round(avg(Year), 3) AS mean, round(stddev(Year), 3) AS std, min(Year) AS min, quantile_cont(Year, 0.01) AS p1, quantile_cont(Year, 0.25) AS p25, quantile_cont(Year, 0.5) AS p50, quantile_cont(Year, 0.75) AS p75, quantile_cont(Year, 0.95) AS p95, quantile_cont(Year, 0.99) AS p99, max(Year) AS max FROM {CLEAN}
UNION ALL SELECT 'Month', count(Month), round(avg(Month), 3), round(stddev(Month), 3), min(Month), quantile_cont(Month, 0.01), quantile_cont(Month, 0.25), quantile_cont(Month, 0.5), quantile_cont(Month, 0.75), quantile_cont(Month, 0.95), quantile_cont(Month, 0.99), max(Month) FROM {CLEAN}
UNION ALL SELECT 'DayofMonth', count(DayofMonth), round(avg(DayofMonth), 3), round(stddev(DayofMonth), 3), min(DayofMonth), quantile_cont(DayofMonth, 0.01), quantile_cont(DayofMonth, 0.25), quantile_cont(DayofMonth, 0.5), quantile_cont(DayofMonth, 0.75), quantile_cont(DayofMonth, 0.95), quantile_cont(DayofMonth, 0.99), max(DayofMonth) FROM {CLEAN}
UNION ALL SELECT 'DayOfWeek', count(DayOfWeek), round(avg(DayOfWeek), 3), round(stddev(DayOfWeek), 3), min(DayOfWeek), quantile_cont(DayOfWeek, 0.01), quantile_cont(DayOfWeek, 0.25), quantile_cont(DayOfWeek, 0.5), quantile_cont(DayOfWeek, 0.75), quantile_cont(DayOfWeek, 0.95), quantile_cont(DayOfWeek, 0.99), max(DayOfWeek) FROM {CLEAN}
UNION ALL SELECT 'Distance', count(Distance), round(avg(Distance), 3), round(stddev(Distance), 3), min(Distance), quantile_cont(Distance, 0.01), quantile_cont(Distance, 0.25), quantile_cont(Distance, 0.5), quantile_cont(Distance, 0.75), quantile_cont(Distance, 0.95), quantile_cont(Distance, 0.99), max(Distance) FROM {CLEAN}
UNION ALL SELECT 'DepDelay', count(DepDelay), round(avg(DepDelay), 3), round(stddev(DepDelay), 3), min(DepDelay), quantile_cont(DepDelay, 0.01), quantile_cont(DepDelay, 0.25), quantile_cont(DepDelay, 0.5), quantile_cont(DepDelay, 0.75), quantile_cont(DepDelay, 0.95), quantile_cont(DepDelay, 0.99), max(DepDelay) FROM {CLEAN}
UNION ALL SELECT 'ArrDelay', count(ArrDelay), round(avg(ArrDelay), 3), round(stddev(ArrDelay), 3), min(ArrDelay), quantile_cont(ArrDelay, 0.01), quantile_cont(ArrDelay, 0.25), quantile_cont(ArrDelay, 0.5), quantile_cont(ArrDelay, 0.75), quantile_cont(ArrDelay, 0.95), quantile_cont(ArrDelay, 0.99), max(ArrDelay) FROM {CLEAN}
UNION ALL SELECT 'CarrierDelay', count(CarrierDelay), round(avg(CarrierDelay), 3), round(stddev(CarrierDelay), 3), min(CarrierDelay), quantile_cont(CarrierDelay, 0.01), quantile_cont(CarrierDelay, 0.25), quantile_cont(CarrierDelay, 0.5), quantile_cont(CarrierDelay, 0.75), quantile_cont(CarrierDelay, 0.95), quantile_cont(CarrierDelay, 0.99), max(CarrierDelay) FROM {CLEAN}
UNION ALL SELECT 'WeatherDelay', count(WeatherDelay), round(avg(WeatherDelay), 3), round(stddev(WeatherDelay), 3), min(WeatherDelay), quantile_cont(WeatherDelay, 0.01), quantile_cont(WeatherDelay, 0.25), quantile_cont(WeatherDelay, 0.5), quantile_cont(WeatherDelay, 0.75), quantile_cont(WeatherDelay, 0.95), quantile_cont(WeatherDelay, 0.99), max(WeatherDelay) FROM {CLEAN}
UNION ALL SELECT 'NASDelay', count(NASDelay), round(avg(NASDelay), 3), round(stddev(NASDelay), 3), min(NASDelay), quantile_cont(NASDelay, 0.01), quantile_cont(NASDelay, 0.25), quantile_cont(NASDelay, 0.5), quantile_cont(NASDelay, 0.75), quantile_cont(NASDelay, 0.95), quantile_cont(NASDelay, 0.99), max(NASDelay) FROM {CLEAN}
UNION ALL SELECT 'SecurityDelay', count(SecurityDelay), round(avg(SecurityDelay), 3), round(stddev(SecurityDelay), 3), min(SecurityDelay), quantile_cont(SecurityDelay, 0.01), quantile_cont(SecurityDelay, 0.25), quantile_cont(SecurityDelay, 0.5), quantile_cont(SecurityDelay, 0.75), quantile_cont(SecurityDelay, 0.95), quantile_cont(SecurityDelay, 0.99), max(SecurityDelay) FROM {CLEAN}
UNION ALL SELECT 'LateAircraftDelay', count(LateAircraftDelay), round(avg(LateAircraftDelay), 3), round(stddev(LateAircraftDelay), 3), min(LateAircraftDelay), quantile_cont(LateAircraftDelay, 0.01), quantile_cont(LateAircraftDelay, 0.25), quantile_cont(LateAircraftDelay, 0.5), quantile_cont(LateAircraftDelay, 0.75), quantile_cont(LateAircraftDelay, 0.95), quantile_cont(LateAircraftDelay, 0.99), max(LateAircraftDelay) FROM {CLEAN}
""")
display(describe)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

3b. Summary statistics (describe)  137.86 s


,column_name,n,mean,std,min,p1,p25,p50,p75,p95,p99,max
0,Year,118503331,1998.633,6.244,1987.0,1987.0,1993.0,1999.0,2004.0,2008.0,2008.0,2008.0
1,Month,118503331,6.564,3.439,1.0,1.0,4.0,7.0,10.0,12.0,12.0,12.0
2,DayofMonth,118503331,15.734,8.791,1.0,1.0,8.0,16.0,23.0,29.0,31.0,31.0
3,DayOfWeek,118503331,3.950,1.991,1.0,1.0,2.0,4.0,6.0,7.0,7.0,7.0
4,Distance,118309103,708.845,554.204,0.0,79.0,309.0,550.0,946.0,1851.0,2553.0,4983.0
5,DepDelay,118503331,8.283,28.518,-1410.0,-10.0,-2.0,0.0,7.0,52.0,129.0,1439.0
6,ArrDelay,118503331,7.258,30.987,-1437.0,-27.0,-7.0,0.0,11.0,57.0,137.0,1438.0
7,CarrierDelay,33063650,3.807,20.086,0.0,0.0,0.0,0.0,0.0,22.0,83.0,1431.0
8,WeatherDelay,33063650,0.815,9.591,0.0,0.0,0.0,0.0,0.0,0.0,23.0,1429.0
9,NASDelay,33063650,4.244,16.858,-60.0,0.0,0.0,0.0,0.0,25.0,79.0,1392.0


In [36]:
categorical = run("3c. Categorical description", f"""
WITH value_counts AS (
    SELECT 'UniqueCarrier' AS column_name, UniqueCarrier AS value, count(*) AS freq FROM {CLEAN} WHERE UniqueCarrier IS NOT NULL GROUP BY 2
    UNION ALL SELECT 'Origin', Origin, count(*) FROM {CLEAN} WHERE Origin IS NOT NULL GROUP BY 2
    UNION ALL SELECT 'Dest', Dest, count(*) FROM {CLEAN} WHERE Dest IS NOT NULL GROUP BY 2
    UNION ALL SELECT 'CancellationCode', CancellationCode, count(*) FROM {CLEAN} WHERE CancellationCode IS NOT NULL GROUP BY 2
)
SELECT column_name,
       sum(freq) AS count,
       count(*) AS n_unique,
       max_by(value, freq) AS top,
       max(freq) AS freq
FROM value_counts
GROUP BY column_name
ORDER BY column_name
""")
display(categorical)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

3c. Categorical description  3.41 s


,column_name,count,n_unique,top,freq
0,CancellationCode,38354965.0,4,,38354955
1,Dest,118503331.0,343,ORD,6355471
2,Origin,118503331.0,347,ORD,6331729
3,UniqueCarrier,118503331.0,29,DL,16073976


In [37]:
t0 = time.perf_counter()

hist = con.execute(f"""
SELECT least(floor((ArrDelay + 60) / 5), 47) * 5 - 60 AS bin_left,
       least(floor((ArrDelay + 60) / 5), 47) * 5 - 55 AS bin_right,
       count(*) AS flights
FROM {CLEAN}
WHERE ArrDelay BETWEEN -60 AND 180
GROUP BY 1, 2
ORDER BY 1
""").df()

pct = con.execute(f"""
SELECT quantile_cont(ArrDelay, [0.01,0.05,0.25,0.5,0.75,0.90,0.95,0.99,0.999]) AS percentiles,
       round(avg(ArrDelay), 3) AS mean_delay,
       round(quantile_cont(ArrDelay, 0.5), 1) AS median_delay,
       count(*) FILTER (WHERE ArrDelay < -60) AS below_window,
       count(*) FILTER (WHERE ArrDelay > 180) AS above_window
FROM {CLEAN}
""").df()

RESULTS.append({"Stage": "4. EDA 1 delay distribution", "Seconds": round(time.perf_counter() - t0, 3)})
display(pct)
display(hist)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,percentiles,mean_delay,median_delay,below_window,above_window
0,"[-27.0, -18.0, -7.0, 0.0, 11.0, 32.0, 57.0, 13...",7.258,0.0,3681,553029


,bin_left,bin_right,flights
0,-60.0,-55.0,3982
1,-55.0,-50.0,9473
2,-50.0,-45.0,23415
3,-45.0,-40.0,58891
4,-40.0,-35.0,149671
5,-35.0,-30.0,378228
6,-30.0,-25.0,950108
7,-25.0,-20.0,2302061
8,-20.0,-15.0,5226842
9,-15.0,-10.0,10410520


In [38]:
by_carrier = run("5. EDA 2 airline comparison", f"""
SELECT UniqueCarrier,
       count(*) AS flights,
       round(avg(ArrDelay), 3) AS mean_delay,
       round(quantile_cont(ArrDelay, 0.5), 1) AS median_delay,
       round(100 * avg(CASE WHEN Delayed15 THEN 1.0 ELSE 0.0 END), 2) AS pct_delayed_15
FROM {CLEAN}
GROUP BY UniqueCarrier
HAVING count(*) >= greatest(1000, 0.0005 * {n_clean})
ORDER BY mean_delay
""")
display(by_carrier)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

5. EDA 2 airline comparison  4.96 s


,UniqueCarrier,flights,mean_delay,median_delay,pct_delayed_15
0,HA,258486,-0.556,-4.0,6.46
1,AQ,145318,1.309,-2.0,9.05
2,ML (1),64177,5.350,0.0,14.42
3,NW,9982197,5.541,-1.0,18.18
4,F9,333680,5.730,0.0,18.85
5,PA (1),294645,5.910,0.0,19.40
6,OO,2942603,6.113,-2.0,17.57
7,9E,503028,6.143,-4.0,18.77
8,TZ,205428,6.166,-3.0,19.05
9,WN,14453463,6.322,0.0,17.73


In [39]:
by_month = run("6. EDA 3 monthly trend", f"""
SELECT Month,
       count(*) AS flights,
       round(avg(ArrDelay), 3) AS mean_delay,
       round(quantile_cont(ArrDelay, 0.5), 1) AS median_delay,
       round(100 * avg(CASE WHEN Delayed15 THEN 1.0 ELSE 0.0 END), 2) AS pct_delayed_15
FROM {CLEAN}
GROUP BY Month
ORDER BY Month
""")
display(by_month)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

6. EDA 3 monthly trend  5.68 s


,Month,flights,mean_delay,median_delay,pct_delayed_15
0,1,9718721,8.665,1.0,22.44
1,2,8974597,8.111,1.0,21.54
2,3,10007561,7.451,0.0,20.40
3,4,9725754,5.438,0.0,17.25
4,5,9966350,5.677,-1.0,17.28
5,6,9841218,9.945,1.0,22.12
6,7,10174710,9.079,0.0,20.90
7,8,10250222,7.974,0.0,20.04
8,9,9464386,3.583,-2.0,14.77
9,10,10378287,4.926,0.0,16.62


In [40]:
causes = run("7. EDA 4 delay cause breakdown", f"""
SELECT Cause, Minutes, round(100 * Minutes / sum(Minutes) OVER (), 2) AS Pct
FROM (
    SELECT 'CarrierDelay' AS Cause, coalesce(sum(CASE WHEN CarrierDelay > 0 THEN CarrierDelay END), 0) AS Minutes FROM {CLEAN}
    UNION ALL
    SELECT 'WeatherDelay', coalesce(sum(CASE WHEN WeatherDelay > 0 THEN WeatherDelay END), 0) FROM {CLEAN}
    UNION ALL
    SELECT 'NASDelay', coalesce(sum(CASE WHEN NASDelay > 0 THEN NASDelay END), 0) FROM {CLEAN}
    UNION ALL
    SELECT 'SecurityDelay', coalesce(sum(CASE WHEN SecurityDelay > 0 THEN SecurityDelay END), 0) FROM {CLEAN}
    UNION ALL
    SELECT 'LateAircraftDelay', coalesce(sum(CASE WHEN LateAircraftDelay > 0 THEN LateAircraftDelay END), 0) FROM {CLEAN}
) t
ORDER BY Minutes DESC
""")
display(causes)

7. EDA 4 delay cause breakdown  0.70 s


,Cause,Minutes,Pct
0,LateAircraftDelay,162671969.0,35.62
1,NASDelay,140309065.0,30.72
2,CarrierDelay,125862581.0,27.56
3,WeatherDelay,26962598.0,5.90
4,SecurityDelay,913344.0,0.20


In [41]:
perf = pd.DataFrame(RESULTS)
total = perf["Seconds"].sum()
perf["Minutes"] = (perf["Seconds"] / 60).round(3)
perf["Pct_of_total"] = (100 * perf["Seconds"] / total).round(1)
perf.loc[len(perf)] = ["TOTAL", round(total, 3), round(total / 60, 3), 100.0]
display(perf)

,Stage,Seconds,Minutes,Pct_of_total
0,0. Disk read benchmark (raw bytes),0.650,0.011,0.1
1,1. Data loading,116.967,1.949,19.1
2,2a. Data quality audit (pre-clean),0.610,0.010,0.1
3,2b. Cleaning,327.170,5.453,53.3
4,3a. Schema description,2.745,0.046,0.4
5,3b. Summary statistics (describe),137.858,2.298,22.5
6,3c. Categorical description,3.407,0.057,0.6
7,4. EDA 1 delay distribution,12.977,0.216,2.1
8,5. EDA 2 airline comparison,4.962,0.083,0.8
9,6. EDA 3 monthly trend,5.685,0.095,0.9


In [42]:
display(con.execute(f"""
SELECT 'rows loaded' AS Item, count(*) AS Value FROM {RAW}
UNION ALL SELECT 'rows kept after cleaning', count(*) FROM {CLEAN}
UNION ALL SELECT 'rows removed', (SELECT count(*) FROM {RAW}) - (SELECT count(*) FROM {CLEAN})
UNION ALL SELECT 'cancelled', count(*) FROM {RAW} WHERE Cancelled = 1
UNION ALL SELECT 'diverted', count(*) FROM {RAW} WHERE Diverted = 1
UNION ALL SELECT 'ArrDelay missing', count(*) FROM {RAW} WHERE ArrDelay IS NULL
""").df())

,Item,Value
0,rows loaded,123534969
1,rows kept after cleaning,118503331
2,rows removed,5031638
3,cancelled,2303324
4,diverted,284204
5,ArrDelay missing,2587529
